# Welcome: your first table

Install Asqueel in a virtual environment, then create, query and change a product table in a Python console. No database files needed.

[Watch lesson 1](https://asqueel.org/learn/01-what-is-asqueel/)

**Run the cells from top to bottom in a fresh Python kernel.** Each notebook includes its own starting state. SQLite runs in memory; restarting the kernel discards the database. To repeat the exercise, restart the kernel and run all cells. Colab execution requires a Google account.

© 2026 Softwell S.r.l. · Apache-2.0

## Before the examples: `_ = ...` is optional

It assigns a return value we do not need, preventing its automatic display in the Python console or notebook. The call still runs normally. This is a Python convention, not an Asqueel requirement. When you need the result, use a meaningful variable name.

## Prepare the notebook

The video starts in a terminal. Here, install the same official package in the notebook kernel. No virtual environment or separate Python console is needed in Colab.

In [ ]:
%pip install -q "asqueel[migration]==0.6.0"

In [ ]:
%config InteractiveShell.ast_node_interactivity = "all"

## 1 — What we'll learn in this lesson

Welcome! Today I'm introducing Asqueel, a Python adapter for SQL.

```text
Welcome to Asqueel
A Python adapter for SQL
```

Here's what we'll learn in this lesson. Install Asqueel, create a product table, write and query data, add columns, and import CSV. We'll work in one Python terminal with SQLite in memory. Closing it discards the data.

```text
What we'll learn in this lesson
Install Asqueel
Create a product table in memory
Insert, query and update
Add columns and migrate
Import CSV data

One Python session · temporary data
```

## 2 — Create a virtual environment

Let's create an isolated environment. Python puts its packages in this dot venv folder.

The terminal step below is already covered by the notebook setup above:

```text
$ python3 -m venv .venv
```

Let's activate it. These are the macOS and Linux commands.

The terminal step below is already covered by the notebook setup above:

```text
$ source .venv/bin/activate
```

Let's install Asqueel. The migration extra includes the engine that creates and changes our tables.

The terminal step below is already covered by the notebook setup above:

```text
(.venv) $ python -m pip install "asqueel[migration]"
```

## 3 — Start Python

Let's open Python. We'll keep this console open for the whole example.

The terminal step below is already covered by the notebook setup above:

```text
(.venv) $ python
```

Let's import our two tools. One describes the model; the other connects it to a database.

In [ ]:
######## Import our two tools ########

from asqueel import AsqueelDb, SqlDatabaseConfig

## 4 — Describe a product table

Let's choose SQLite in memory. This model lives right here in Python. No model file or class is needed.

In [ ]:
######## Choose SQLite in memory ########

model = SqlDatabaseConfig()
root = model.source.db(implementation="sqlite")
_ = root.connection(name=":memory:", implementation="sqlite")

Let's describe a product table. It needs a code and a name. Code is the unique key; both columns contain text.

In [ ]:
######## Describe a product table ########

shop = root.packages().package("shop", sqlschema="shop")
product = shop.table("product", pkey="code")
_ = product.column("code", size=":10")
_ = product.column("name", size=":60")

## 5 — Create the table

Let's create that table. Migrate compares the model with the database and runs the missing SQL. Here is the create table statement.

In [ ]:
######## Create that table ########

db = AsqueelDb(model)
print(db.migrate().commands)

## 6 — Insert a product

Let's add our first product. Insert tea, then commit the transaction. The data stays in memory, not on disk.

In [ ]:
######## Add our first product ########

products = db.table("shop.product")
products.insert({"code": "TEA", "name": "Tea"})
db.commit()

## 7 — Read and update

Let's read our product. Select its code and name, then fetch the result.

In [ ]:
######## Read our product ########

products.query(columns="$code, $name").fetch()

Let's change its name. Find the row by its key, set Green tea, and commit.

In [ ]:
######## Change its name ########

products.update({"name": "Green tea"}, pkey="TEA")
db.commit()

Let's check the update. The same query now returns Green tea.

In [ ]:
######## Check the update ########

products.query(columns="$code, $name").fetch()
db.commit()

## 8 — Add two columns

Let's track category and stock. Add a text column for category and an integer column for stock.

In [ ]:
######## Track category and stock ########

_ = product.column("category", size=":30")
_ = product.column("stock", dtype="I")

Let's load the updated model. Reload changes the running model while keeping our database connection.

In [ ]:
######## Load the updated model ########

db.reloadModel(model)

## 9 — Migrate again

Let's add those database columns. Migrate now generates two add column statements, without recreating the table.

In [ ]:
######## Add those database columns ########

print(db.migrate().commands)

Let's check our original row. Get a fresh table handle. Green tea is still there; stock has no value yet.

In [ ]:
######## Check our original row ########

products = db.table("shop.product")
products.query(columns="$name, $stock").fetch()
db.commit()

## 10 — Write the new fields

Let's give tea its stock. Set its category to Drinks and its stock to eight, then commit.

In [ ]:
######## Give tea its stock ########

products.update({"category": "Drinks", "stock": 8}, pkey="TEA")
db.commit()

Let's read the new values. The query includes our two new columns.

In [ ]:
######## Read the new values ########

products.query(columns="$name, $category, $stock").fetch()
db.commit()

## 11 — Import CSV data

Let's add products from CSV. For this example, keep the CSV in a Python string too. Its headers match the table columns.

In [ ]:
######## Add products from CSV ########

from io import StringIO
data = StringIO(
    "code,name,category,stock\n"
    "COF,Coffee,Drinks,24\n"
    "MUG,Mug,Kitchen,12\n"
)

Let's import those rows. Pass the stream to SQL insert and specify CSV. Asqueel converts stock using the model and returns the imported row count. Then commit.

In [ ]:
######## Import those rows ########

products.sql_insert(data, format="csv")
db.commit()

## 12 — Query the imported rows

Let's list just the drinks. Use a parameter for the category and sort by stock, largest first. Coffee from the CSV, then our Green tea.

In [ ]:
######## List just the drinks ########

products.query(
    columns="$name, $stock",
    where="$category = :category", category="Drinks",
    order_by="$stock DESC",
).fetch()
db.commit()

## 13 — Thank you — next: customers

Thanks for watching! Next, we'll add a customer table and follow the relationships in a small invoicing example.

```text
Thank you for following along!
Next lesson: add a customer table
Build invoices and follow relationship paths
```

## Check your result

Run this final cell to check the exercise.

In [ ]:
assert products.query().count() == 3
assert products.query(columns="$stock", where="$code = :code", code="TEA").fetch()[0]["stock"] == 8
db.commit()
print("Lesson checks passed.")
